# Random quadratic attractor screening notebook

This notebook is the slower companion to `python/random_quadratic_attractor_screening.py` and `art/random-quadratic-attractor-screening.csv`. The point is not to claim a theorem about chaos from one quick screen. The point is narrower: show how a bounded search can separate exploding maps, trivial bounded maps, and a smaller pool of survivors worth looking at.


## Question and scope

Question: if we sample hundreds of quadratic maps in two variables, do enough of them survive a simple bounded screen to make a public artifact that teaches something?

Scope boundary:
- this notebook reads the generated CSV from one deterministic scan
- it uses a finite-step Lyapunov-style proxy and a coarse occupancy score
- it does **not** prove that every accepted survivor is a mathematically certified strange attractor
- it does **not** try to be a general nonlinear-dynamics package


## Map family

The sampled family is

`x_{n+1} = a_0 + a_1 x + a_2 x^2 + a_3 x y + a_4 y + a_5 y^2`

`y_{n+1} = b_0 + b_1 x + b_2 x^2 + b_3 x y + b_4 y + b_5 y^2`

with each coefficient drawn uniformly from `[-1.2, 1.2]` under a fixed random seed. Each candidate gets:
- a warmup run to discard transients,
- a sampled orbit window,
- a largest-Lyapunov-style proxy from the Jacobian along that window,
- and a coarse occupancy score on a `40 x 40` grid over the sampled bounding box.


In [ ]:
import csv
from collections import Counter
from pathlib import Path

csv_path = Path('../art/random-quadratic-attractor-screening.csv')
rows = list(csv.DictReader(csv_path.open()))
len(rows), rows[0].keys()


In [ ]:
status_counts = Counter(row['status'] for row in rows)
status_counts


## What the screen actually did

This run is a nice reminder that the search itself is the artifact. Most coefficient sets fail fast. That is not wasted effort. It is the evidence that the accepted structures were not free.

The rules were:
1. reject any orbit that diverges beyond `|x|` or `|y| > 50`,
2. reject bounded runs whose sampled box is tiny or whose coarse occupancy stays below `3%`,
3. reject bounded runs whose finite-step Lyapunov-style proxy stays at or below `0.02`,
4. rank the surviving pool by `score = lambda_max * occupancy`, then keep the top six for the card.


In [ ]:
selected = [row for row in rows if row['selected_rank']]
selected = sorted(selected, key=lambda row: int(row['selected_rank']))
[(row['selected_rank'], row['candidate_id'], float(row['lambda_max']), float(row['occupancy']), float(row['score'])) for row in selected]


## Reading the chosen survivors

The six chosen maps do not all win the same way:
- some survive on higher occupancy and look like richer filled webs,
- some survive on a stronger Lyapunov-style proxy even when their occupied area is thinner,
- the score is just a compact way to keep both ideas in the room at once.

That makes the public card more honest than a gallery of hand-picked curiosities. It shows that the survivors came from a bounded filter, not from aesthetic cherry-picking alone.


In [ ]:
accepted = [row for row in rows if row['status'] == 'accepted']
accepted_lambdas = [float(row['lambda_max']) for row in accepted]
accepted_occupancies = [float(row['occupancy']) for row in accepted]
{
    'accepted_pool_size': len(accepted),
    'lambda_min': min(accepted_lambdas),
    'lambda_median': sorted(accepted_lambdas)[len(accepted_lambdas)//2],
    'lambda_max': max(accepted_lambdas),
    'occupancy_min': min(accepted_occupancies),
    'occupancy_median': sorted(accepted_occupancies)[len(accepted_occupancies)//2],
    'occupancy_max': max(accepted_occupancies),
}


## Caveat

A finite-step Lyapunov estimate is useful, but it is still a screening tool. It can separate obvious losers from bounded interesting candidates, yet it does not replace a deeper stability analysis, invariant-measure study, or attractor classification argument.

That caveat is part of why this lane works for the repo: it is a compact measurement story, not fake mathematical certainty.


## References and next moves

- Paul Bourke, *Random Attractors - Found using Lyapunov Exponents*
- Eric Weisstein, *Chaos Game*, MathWorld
- Juan Carlos Ponce Campuzano, *Strange Attractors*

Natural follow-ups, if the repo wants one:
- compare the same screening rule under a tighter coefficient box,
- test whether the selected pool changes under a longer sample horizon,
- or add one second score that penalizes extremely thin survivors without pretending there is one universal beauty metric.
